In [8]:
from __future__ import annotations

import json
import zipfile
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score, 
    recall_score
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

In [9]:
DATA_PATH = './data/datasets/sinistros/datatran2025.csv'

In [10]:
# Funções auxiliares para manipular o dataset

def load_dataset(path: Path, sep: str = None, encoding: str = "latin1") -> pd.DataFrame:
    df = pd.read_csv(path, sep=sep, engine="python", encoding=encoding)
    return df


def preprocess_dataset(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    for column in [
        "mortos",
        "feridos_leves",
        "feridos_graves",
        "ilesos",
        "feridos",
        "ignorados",
    ]:
        df[f"teve_{column}"] = df[column] > 0

    return df

# Funções auxiliares para trabalhar com os dados de treino e teste

# Adicionar o controle de sparse para evitar erros do Iterative e do Knn imputers, relacionados ao recebimento de 
# dados esparsos, que quebrava a execução dos pipelines.
def make_one_hot_encoder(sparse: bool = True) -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=sparse)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=sparse)

def split_X_y(df: pd.DataFrame, target: str, extra: list[str]) -> tuple[pd.DataFrame, pd.Series]:
    if target not in df.columns:
        raise ValueError(f"Target '{target}' nao esta no dataset.")
    return df.drop(columns=[target] + extra), df[target]

# Tokens que representam valores ausentes
NUMERIC_MISSING_VALUES = [-200, -200.0]

STRING_MISSING_VALUES = [
    "?", " ?", "? ", "NA", "N/A", "na", "n/a", "NaN", "nan", "", " ",
    "unknown", "Unknown", "-200",
]

def normalize_missing_values(df: pd.DataFrame) -> pd.DataFrame:
    """Substitui tokens de missing por NaN, coluna a coluna, respeitando o dtype.
    
    - Colunas numericas: substitui -200 e -200.0
    - Colunas de texto: faz strip e substitui tokens como '?', 'unknown', etc.
    Essa abordagem evita erros do pandas 2.x ao misturar int e str no replace().
    -- TODO: verificar se tem outros simbolos estranhos nos dataset da atividade (pedir par aos alunos checarem isso)
    """
    df = df.copy()
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].replace(NUMERIC_MISSING_VALUES, np.nan)
        else:
            # Normaliza os tokens de ausente para np.nan antes do imputador,
            # porque pd.NA nesta base gera erro no pipeline com scikit-learn/pandas.
            # Exemplo do SimpleImputer: "boolean value of NA is ambiguous"
            df[col] = df[col].astype(object)
            df[col] = df[col].apply(lambda v: v.strip() if isinstance(v, str) else v)
            df[col] = df[col].replace(STRING_MISSING_VALUES, np.nan)
    return df



# Pré-Processamento

In [11]:
df = preprocess_dataset(load_dataset(DATA_PATH))

df

,id,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,...,longitude,regional,delegacia,uop,teve_mortos,teve_feridos_leves,teve_feridos_graves,teve_ilesos,teve_feridos,teve_ignorados
0,652493,2025-01-01,quarta-feira,06:20:00,SP,116,225,GUARULHOS,Reação tardia ou ineficiente do condutor,Tombamento,...,"-46,54075317",SPRF-SP,DEL01-SP,UOP01-DEL01-SP,False,True,False,False,True,True
1,652519,2025-01-01,quarta-feira,07:50:00,CE,116,"546,2",PENAFORTE,Pista esburacada,Colisão frontal,...,"-39,08333306",SPRF-CE,DEL05-CE,UOP03-DEL05-CE,True,True,False,True,True,True
2,652522,2025-01-01,quarta-feira,08:45:00,PR,369,"88,2",CORNELIO PROCOPIO,Reação tardia ou ineficiente do condutor,Colisão traseira,...,"-50,637228",SPRF-PR,DEL07-PR,UOP05-DEL07-PR,False,True,False,True,True,False
3,652544,2025-01-01,quarta-feira,11:00:00,PR,116,74,CAMPINA GRANDE DO SUL,Reação tardia ou ineficiente do condutor,Saída de leito carroçável,...,"-49,04223028",SPRF-PR,DEL01-PR,UOP02-DEL01-PR,False,True,False,True,True,False
4,652549,2025-01-01,quarta-feira,09:30:00,MG,251,471,FRANCISCO SA,Velocidade Incompatível,Colisão frontal,...,"-43,43121303",SPRF-MG,DEL12-MG,UOP01-DEL12-MG,False,True,True,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72524,757462,2025-12-20,sábado,13:10:00,MG,381,"279,5",JAGUARACU,Acessar a via sem observar a presença dos outr...,Colisão transversal,...,"-42,7611",SPRF-MG,DEL03-MG,UOP02-DEL03-MG,False,True,False,True,True,False
72525,757492,2025-09-27,sábado,13:50:00,SC,282,381,HERVAL DOESTE,Conversão proibida,Colisão transversal,...,"-51,50077",SPRF-SC,DEL07-SC,UOP05-DEL07-SC,False,True,True,False,True,True
72526,757593,2025-12-14,domingo,13:50:00,PE,104,62,CARUARU,Ausência de reação do condutor,Colisão transversal,...,"-35,97546",SPRF-PE,DEL02-PE,UOP01-DEL02-PE,False,True,False,True,True,False
72527,758175,2025-12-15,segunda-feira,15:50:00,SC,101,130,CAMBORIU,Condutor deixou de manter distância do veículo...,Colisão traseira,...,"-48,6609548",SPRF-SC,DEL04-SC,UOP03-DEL04-SC,False,False,False,True,False,False


In [12]:
boolean_cols = df.select_dtypes(include="bool").columns.tolist()

bool_counts = pd.DataFrame(
    {
        "True": df[boolean_cols].sum(),
        "False": (~df[boolean_cols]).sum(),
    }
).astype(int)

display(bool_counts)

categorical_columns = [
    "tipo_acidente",
    "classificacao_acidente",
    "fase_dia",
    "causa_acidente",
    "condicao_metereologica",
    "tipo_pista",
    "tracado_via",
    "regional"
]

unique_values = {
    column: sorted(df[column].dropna().unique().tolist())
    for column in categorical_columns
    if column in df.columns
}

unique_values_df = pd.DataFrame(
    {
        "coluna": list(unique_values.keys()),
        "valores_unicos": list(unique_values.values()),
    }
 )

display(unique_values_df)

,True,False
teve_mortos,5210,67319
teve_feridos_leves,45966,26563
teve_feridos_graves,16354,56175
teve_ilesos,45198,27331
teve_feridos,58042,14487
teve_ignorados,19857,52672


,coluna,valores_unicos
0,tipo_acidente,"[Atropelamento de Animal, Atropelamento de Ped..."
1,classificacao_acidente,"[Com Vítimas Fatais, Com Vítimas Feridas, Sem ..."
2,fase_dia,"[Amanhecer, Anoitecer, Plena Noite, Pleno dia]"
3,causa_acidente,[Acessar a via sem observar a presença dos out...
4,condicao_metereologica,"[Chuva, Céu Claro, Garoa/Chuvisco, Ignorado, N..."
5,tipo_pista,"[Dupla, Múltipla, Simples]"
6,tracado_via,"[Aclive, Aclive;Curva, Aclive;Curva;Em Obras, ..."
7,regional,"[SPRF-AC, SPRF-AL, SPRF-AM, SPRF-AP, SPRF-BA, ..."


# Teste baseline com dropna

Para testar o desempenho do dataset sem nenhum pré-processamento mais intensivo e integração com fontes externas, será criado um baseline com dropna.

In [13]:

def build_preprocessor_baseline(X: pd.DataFrame, scale_numeric: bool) -> ColumnTransformer:
    numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = [c for c in X.columns if c not in numeric_cols]
    return ColumnTransformer(
        transformers=[
            ("num", StandardScaler() if scale_numeric else "passthrough", numeric_cols),
            ("cat", make_one_hot_encoder(), categorical_cols),
        ],
        remainder="drop",
    )


def run_classification_baseline(X: pd.DataFrame, y: pd.Series, test_size: float, random_state: int) -> dict:
    class_counts = y.value_counts(dropna=False)
    stratify = y if class_counts.min() >= 2 else None
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=stratify
    )

    models = {
        "knn": Pipeline([
            ("preprocess", build_preprocessor_baseline(X_train, scale_numeric=True)),
            ("model", KNeighborsClassifier(n_neighbors=5, n_jobs=-1)),
        ]),
        "random_forest": Pipeline([
            ("preprocess", build_preprocessor_baseline(X_train, scale_numeric=False)),
            ("model", RandomForestClassifier(
                n_estimators=200,
                random_state=random_state,
                n_jobs=-1,
                class_weight="balanced_subsample",
            )),
        ]),
    }

    results = {}
    for name, model in models.items():
        model.fit(X_train, y_train)
        pred = model.predict(X_test)

        metrics = {
            "accuracy": float(accuracy_score(y_test, pred)),
            "macro_f1": float(f1_score(y_test, pred, average="macro")),
            "weighted_f1": float(f1_score(y_test, pred, average="weighted")),
            # Como o weighted está sendo usado para escolher o melhor modelo, vou coletar o precision e o recall para ter uma noção melhor do porquê o f1 deu o valor que deu
            "weighted_precision": float(precision_score(y_test, pred, average="weighted")),
            "weighted_recall": float(recall_score(y_test, pred, average="weighted")),
        }
        results[name] = metrics

        print(f"\n[model] {name}")
        print(metrics)
        print("Matriz de confusao:")
        print(confusion_matrix(y_test, pred))
        print(classification_report(y_test, pred))

    return results


def run_one_dataset_with_drop(key: str, target, out_dir: str = "data", test_size: float = 0.2, random_state: int = 42) -> dict:
    data_dir = Path(out_dir)
    df = preprocess_dataset(load_dataset(data_dir / f"{key}.csv"))

    print("\n" + "=" * 80)
    print(f"Dataset: {key} | Target: {target}")
    print("=" * 80)
    print("Shape original:", df.shape)

    df = normalize_missing_values(df)
    
    rows_before = len(df)
    df_clean = df.dropna(axis=0, how="any").copy()
    rows_after = len(df_clean)

    rows_dropped = rows_before - rows_after

    print("Linhas removidas no dropna:", rows_dropped, f"({(rows_dropped/rows_before):.2%})")

    if rows_after == 0:
        raise RuntimeError("Dataset vazio apos dropna.")

    X, y = split_X_y(df_clean, target, ['mortos', 'delegacia', 'uop', 'teve_mortos', 'teve_feridos_leves', 'teve_feridos_graves', 'teve_ilesos', 'teve_feridos', 'teve_ignorados', 'feridos', 'feridos_leves', 'feridos_graves', 'ilesos', 'feridos', 'ignorados'])

    print("Distribuicao da target:")
    print(y.value_counts())
    results = run_classification_baseline(X, y, test_size=test_size, random_state=random_state)

    result_dir = data_dir / "results" / "baseline"
    result_dir.mkdir(parents=True, exist_ok=True)
    out_path = result_dir / f"{key}_baseline_results.json"
    out_path.write_text(
        json.dumps(
            {
                "key": key,
                "target": target,
                "variant": "default",
                "shape_original": list(df.shape),
                "shape_after_prepro": list(df_clean.shape),
                "test_size": test_size,
                "random_state": random_state,
                "results": results,
            },
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )
    print("Resultados salvos em:", out_path)

    return {
        "key": key,
        "strategy": "dropna",
        "variant": "default",
        "rows_original": int(df.shape[0]),
        "rows_after_prepros": int(df_clean.shape[0]),
        "results": results,
    }

In [14]:

print(f"{"=" * 5} DROPNA {"=" * 5}")

dataset="datasets/sinistros/datatran2025"
target="teve_mortos"
out_dir = "data"
test_size = 0.2
random_state = 42

all_runs = []
try:
    run_info = run_one_dataset_with_drop(
        key=dataset,
        target=target,
        out_dir=out_dir,
        test_size=test_size,
        random_state=random_state,
    )
    all_runs.append(run_info)
except Exception as e:
    print(f"\n[ERRO] key={dataset}: {e}")
    all_runs.append({"key": dataset, "error": str(e)})
    


# --- Monta tabelas separadas por tipo de tarefa ---

classification_rows = []
regression_rows = []

for r in all_runs:
    base = {"experiment": "dropna", "key": r["key"], "variant": r.get("variant", "default"), "best_model": "", "error": r.get("error", "")}

    if "error" in r and "results" not in r:
        # Dataset falhou: adiciona na tabela correta com metricas vazias
        classification_rows.append({**base, "accuracy": None, "macro_f1": None, "weighted_f1": None, "weighted_precision": None, "weighted_recall": None})
        continue

    best = max(r["results"], key=lambda m: r["results"][m]["weighted_f1"])
    m = r["results"][best]
    classification_rows.append({
        "experiment": "dropna",
        "key": r["key"],
        "variant": r["variant"],
        "best_model": best,
        "accuracy": round(m["accuracy"], 4),
        "macro_f1": round(m["macro_f1"], 4),
        "weighted_f1": round(m["weighted_f1"], 4),
        "weighted_precision": round(m["weighted_precision"], 4),
        "weighted_recall": round(m["weighted_recall"], 4),
        "error": "",
    })

# Exibe  resultados e salva CSVs 

data_path = Path(out_dir)

if classification_rows:
    clf_df = pd.DataFrame(classification_rows, columns=["experiment", "key", "variant", "best_model", "accuracy", "macro_f1", "weighted_f1", "weighted_precision", "weighted_recall", "error"])
    print("\n=== Classificacao ===")
    display(clf_df)
    clf_path = data_path / "classification_summary.csv"
    clf_df.to_csv(clf_path, index=False)
    print("Salvo em:", clf_path)


print(f"{"=" * 5} FIM DROPNA {"=" * 5}")

===== DROPNA =====

Dataset: datasets/sinistros/datatran2025 | Target: teve_mortos
Shape original: (72529, 36)
Linhas removidas no dropna: 39 (0.05%)
Distribuicao da target:
teve_mortos
False    67281
True      5209
Name: count, dtype: int64

[model] knn
{'accuracy': 0.9629604083321838, 'macro_f1': 0.8179881589131488, 'weighted_f1': 0.9570781584716851, 'weighted_precision': 0.9639154861019226, 'weighted_recall': 0.9629604083321838}
Matriz de confusao:
[[13450     6]
 [  531   511]]
              precision    recall  f1-score   support

       False       0.96      1.00      0.98     13456
        True       0.99      0.49      0.66      1042

    accuracy                           0.96     14498
   macro avg       0.98      0.74      0.82     14498
weighted avg       0.96      0.96      0.96     14498


[model] random_forest
{'accuracy': 0.9832390674575804, 'macro_f1': 0.9295289449756867, 'weighted_f1': 0.9822078942124907, 'weighted_precision': 0.9835363816124681, 'weighted_recall': 0.

,experiment,key,variant,best_model,accuracy,macro_f1,weighted_f1,weighted_precision,weighted_recall,error
0,dropna,datasets/sinistros/datatran2025,default,,None,None,None,None,None,[Errno 2] Arquivo ou diretório inexistente: 'd...


Salvo em: data/classification_summary.csv
===== FIM DROPNA =====


# Integração com fontes externas

## Inmet

A primeira fonte de integração externa a ser utilizada será o INMET, que fornecerá dados acerca das condições climáticas no momento do acidente.

Os datasets do INMET, tem algumas peculiaridades que dificultam sua integração:

**Cabeçalho nos CSVs:**

```txt
REGIAO:;CO
UF:;DF
ESTACAO:;BRASILIA
CODIGO (WMO):;A001
LATITUDE:;-15,78944444
LONGITUDE:;-47,92583332
ALTITUDE:;1160,96
DATA DE FUNDACAO:;07/05/00
<Colunas>
```

Todo csv do INMET apresenta seus dados da estação nas 8 primeiras linhas do dataset, então, não se pode simplesmente só ler o arquivo e extrair as informações úteis. A partir da 9ª linha, começam os dados históricos da estação naquele no ano em escolhido.

**Grande quantidade de arquivos**

Para cada estação existe um arquivo, totalizando mais de quinhentos arquivos. Mesmo com um grande volume de acidentes, eles ocorreram em rodovias federais, então, é altamente improvável que todas as estações sejam usadas para complementar os dados de acidentes.


---

Para resolver esses problemas, é necessário realizar alguns pré-processamentos:
1. Extrair todos os dados das estações em um arquivo separado
2. Com as estações, identificar quais são efetivamente viáveis para complementar o dataset de acidentes, descartando as que forem desnecessárias.
3. Das estações úteis, condensar todo o histórico dessas estações em um único CSV.
4. O objetivo é transformar todos os dados do INMET em dois CSVs: estações e histórico estações.



In [24]:
from pathlib import Path
import re

import pandas as pd

INMET_YEARS = [2025]  # edite esta lista para incluir novos anos
INMET_RAW_ROOT = Path("./data/datasets/inmet_raw")
INMET_PROCESSED_ROOT = Path("./data/datasets/inmet_processed")
INMET_FILENAME_RE = re.compile(
    r"^INMET_(?P<regiao>[^_]+)_(?P<uf>[^_]+)_(?P<estacao_id>[^_]+)_(?P<cidade>.+)_(?P<periodo>\d{2}-\d{2}-\d{4}_A_\d{2}-\d{2}-\d{4})\.CSV$",
    re.IGNORECASE,
)


def normalize_inmet_token(value: str) -> str:
    return re.sub(r"\s+", "-", value.strip())



def parse_inmet_filename(filename: str) -> dict[str, str]:
    match = INMET_FILENAME_RE.match(filename)
    if match is None:
        raise ValueError(f"Nome de arquivo INMET fora do padrao esperado: {filename}")

    info = match.groupdict()
    return {
        "regiao": info["regiao"].upper(),
        "uf": info["uf"].upper(),
        "estacao_id": info["estacao_id"].upper(),
        "cidade": normalize_inmet_token(info["cidade"]),
        "periodo": info["periodo"],
    }



def build_inmet_output_stem(filename: str) -> str:
    info = parse_inmet_filename(filename)
    return (
        f"inmet_{info['regiao']}_{info['uf']}_{info['estacao_id']}_"
        f"{info['cidade']}_{info['periodo']}_dados"
    )



def load_inmet_station_metadata(path: Path) -> pd.DataFrame:
    metadata = pd.read_csv(
        path,
        sep=";",
        encoding="latin1",
        nrows=8,
        header=None,
        names=["coluna", "valor"],
        engine="python",
    )
    metadata["coluna"] = metadata["coluna"].astype(str).str.replace(":", "", regex=False).str.strip()
    metadata["valor"] = metadata["valor"].astype(str).str.strip()
    return metadata



def load_inmet_data(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, sep=";", encoding="latin1", skiprows=8, engine="python")
    df = df.dropna(axis=1, how="all")
    df.columns = [str(column).strip() for column in df.columns]
    return df



def preprocess_inmet_file(path: Path, output_dir: Path) -> dict[str, str]:
    output_dir.mkdir(parents=True, exist_ok=True)

    output_stem = build_inmet_output_stem(path.name)
    metadata_path = output_dir / f"{output_stem}_station_metadata.csv"
    data_path = output_dir / f"{output_stem}.csv"

    metadata_df = load_inmet_station_metadata(path)
    data_df = load_inmet_data(path)

    metadata_df.to_csv(metadata_path, index=False)
    data_df.to_csv(data_path, index=False)

    return {
        "input_file": path.name,
        "metadata_path": str(metadata_path),
        "data_path": str(data_path),
    }



def preprocess_inmet_directory(year: int | str, source_root: Path, output_root: Path) -> list[dict[str, str]]:
    year_label = str(year)
    source_dir = source_root / year_label
    output_dir = output_root / year_label

    if not source_dir.exists():
        raise FileNotFoundError(f"Diretorio INMET nao encontrado: {source_dir}")

    csv_files = sorted(
        path for path in source_dir.iterdir()
        if path.is_file() and path.suffix.lower() == ".csv"
    )

    results: list[dict[str, str]] = []
    for path in csv_files:
        try:
            item = preprocess_inmet_file(path, output_dir)
            item["year"] = year_label
            results.append(item)
        except Exception as exc:
            print(f"[ERRO] {path.name}: {exc}")

    print(f"Processados {len(results)} de {len(csv_files)} arquivos INMET em {output_dir}")
    return results


inmet_results = []
for year in INMET_YEARS:
    try:
        inmet_results.extend(preprocess_inmet_directory(year, INMET_RAW_ROOT, INMET_PROCESSED_ROOT))
    except Exception as exc:
        print(f"[ERRO] ano={year}: {exc}")

inmet_results[:3]


Processados 594 de 594 arquivos INMET em data/datasets/inmet_processed/2025


[{'input_file': 'INMET_CO_DF_A001_BRASILIA_01-01-2025_A_31-12-2025.CSV',
  'metadata_path': 'data/datasets/inmet_processed/2025/inmet_CO_DF_A001_BRASILIA_01-01-2025_A_31-12-2025_dados_station_metadata.csv',
  'data_path': 'data/datasets/inmet_processed/2025/inmet_CO_DF_A001_BRASILIA_01-01-2025_A_31-12-2025_dados.csv',
  'year': '2025'},
 {'input_file': 'INMET_CO_DF_A042_BRAZLANDIA_01-01-2025_A_31-12-2025.CSV',
  'metadata_path': 'data/datasets/inmet_processed/2025/inmet_CO_DF_A042_BRAZLANDIA_01-01-2025_A_31-12-2025_dados_station_metadata.csv',
  'data_path': 'data/datasets/inmet_processed/2025/inmet_CO_DF_A042_BRAZLANDIA_01-01-2025_A_31-12-2025_dados.csv',
  'year': '2025'},
 {'input_file': 'INMET_CO_DF_A045_AGUAS EMENDADAS_01-01-2025_A_31-12-2025.CSV',
  'metadata_path': 'data/datasets/inmet_processed/2025/inmet_CO_DF_A045_AGUAS-EMENDADAS_01-01-2025_A_31-12-2025_dados_station_metadata.csv',
  'data_path': 'data/datasets/inmet_processed/2025/inmet_CO_DF_A045_AGUAS-EMENDADAS_01-01-2025_

In [ ]:
from pathlib import Path
import re
import unicodedata
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd

INMET_YEARS = [2025]
INMET_PROCESSED_ROOT = Path("./data/datasets/inmet_processed")
INMET_AGGREGATED_ROOT = Path("./data/datasets/inmet_aggregated")
ESTACOES_BASE_PATH = INMET_AGGREGATED_ROOT / "estacoes.csv"
ESTACOES_FINAIS_PATH = INMET_AGGREGATED_ROOT / "estacoes_finais.csv"
SINISTRO_ESTACAO_MAIS_PROXIMA_PATH = INMET_AGGREGATED_ROOT / "sinistro_estacao_mais_proxima.csv"
INMET_HISTORICO_PATH = INMET_AGGREGATED_ROOT / "inmet_historico.csv"
SINISTROS_SOURCE_PATH = Path("./data/datasets/sinistros/datatran2025.csv")
BRASILIA_TZ = ZoneInfo("America/Sao_Paulo")

METADATA_KEY_ALIASES = {
    "regiao": "regiao",
    "uf": "uf",
    "estacao": "estacao",
    "codigo_wmo": "wmo",
    "wmo": "wmo",
    "latitude": "latitude",
    "longitude": "longitude",
    "altitude": "altitude",
    "data_de_fundacao": "data_de_fundacao",
}

EXPECTED_STATION_COLUMNS = [
    "wmo",
    "regiao",
    "uf",
    "estacao",
    "latitude",
    "longitude",
    "altitude",
    "data_de_fundacao",
    "source_file",
]

EXPECTED_MAPPING_COLUMNS = ["id", "wmo", "distancia_km"]


def slugify_text(value: object) -> str:
    """Normaliza texto para lower_snake_case ASCII."""
    text = unicodedata.normalize("NFKD", str(value))
    text = text.encode("ascii", "ignore").decode("ascii")
    text = text.lower().strip()
    return re.sub(r"[^a-z0-9]+", "_", text).strip("_")


def normalize_dataframe_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Padroniza os nomes das colunas para lower_snake_case ASCII."""
    kept_columns: list[str] = []
    normalized_columns: list[str] = []
    counts: dict[str, int] = {}

    for column in df.columns:
        normalized = slugify_text(column)
        if not normalized or normalized.startswith("unnamed"):
            continue

        count = counts.get(normalized, 0)
        counts[normalized] = count + 1
        kept_columns.append(column)
        normalized_columns.append(normalized if count == 0 else f"{normalized}_{count + 1}")

    cleaned_df = df.loc[:, kept_columns].copy()
    cleaned_df.columns = normalized_columns
    return cleaned_df


def normalize_station_value(value: object) -> str | None:
    """Remove espaços e trata strings vazias como ausentes."""
    if value is None or pd.isna(value):
        return None

    text = str(value).strip()
    if not text or text.lower() in {"nan", "none"}:
        return None
    return text


def clean_station_catalog(df: pd.DataFrame) -> pd.DataFrame:
    """Limpa o catálogo sem alterar o schema esperado."""
    cleaned_df = normalize_dataframe_columns(df)
    for column in cleaned_df.columns:
        cleaned_df[column] = cleaned_df[column].map(normalize_station_value)
    return cleaned_df


def load_station_metadata_row(metadata_path: Path) -> dict[str, str | None]:
    """Lê um arquivo de metadados do INMET e extrai os campos conhecidos."""
    metadata_df = pd.read_csv(metadata_path, dtype=str)

    if not {"coluna", "valor"}.issubset(metadata_df.columns):
        raise ValueError(f"Metadata invalido em {metadata_path.name}")

    metadata_lookup: dict[str, str | None] = {}
    for _, row in metadata_df[["coluna", "valor"]].iterrows():
        raw_key = slugify_text(row["coluna"])
        mapped_key = METADATA_KEY_ALIASES.get(raw_key)
        if mapped_key is not None:
            metadata_lookup[mapped_key] = normalize_station_value(row["valor"])

    wmo = metadata_lookup.get("wmo")
    if not wmo:
        raise ValueError(f"Nao foi possivel extrair o WMO de {metadata_path.name}")

    return {
        "wmo": wmo,
        "regiao": metadata_lookup.get("regiao"),
        "uf": metadata_lookup.get("uf"),
        "estacao": metadata_lookup.get("estacao"),
        "latitude": metadata_lookup.get("latitude"),
        "longitude": metadata_lookup.get("longitude"),
        "altitude": metadata_lookup.get("altitude"),
        "data_de_fundacao": metadata_lookup.get("data_de_fundacao"),
        "source_file": metadata_path.name.replace("_station_metadata.csv", ".csv"),
    }


def build_station_catalog_from_processed(years: list[int | str]) -> pd.DataFrame:
    """Monta o catálogo base a partir dos metadados já processados."""
    station_rows: list[dict[str, str | None]] = []
    seen_wmo: set[str] = set()

    for year in years:
        year_dir = INMET_PROCESSED_ROOT / str(year)
        if not year_dir.exists():
            print(f"[AVISO] Diretorio ausente, ignorando: {year_dir}")
            continue

        metadata_files = sorted(year_dir.glob("*_station_metadata.csv"))
        if not metadata_files:
            print(f"[AVISO] Nenhum metadata encontrado em: {year_dir}")
            continue

        for metadata_path in metadata_files:
            try:
                station_row = load_station_metadata_row(metadata_path)
            except Exception as exc:
                print(f"[ERRO] {metadata_path.name}: {exc}")
                continue

            wmo = station_row["wmo"]
            if wmo in seen_wmo:
                continue

            station_rows.append(station_row)
            seen_wmo.add(wmo)

    if not station_rows:
        raise RuntimeError("Nenhum arquivo INMET valido foi encontrado para consolidacao das estacoes.")

    station_df = pd.DataFrame(station_rows, columns=EXPECTED_STATION_COLUMNS)
    return clean_station_catalog(station_df)


def load_station_catalog() -> pd.DataFrame:
    """Carrega o catálogo base ou o recompõe por processamento local."""
    if ESTACOES_BASE_PATH.exists():
        print(f"Cache hit: {ESTACOES_BASE_PATH}")
        return clean_station_catalog(pd.read_csv(ESTACOES_BASE_PATH, dtype=str))[EXPECTED_STATION_COLUMNS]

    station_df = build_station_catalog_from_processed(INMET_YEARS)
    INMET_AGGREGATED_ROOT.mkdir(parents=True, exist_ok=True)
    station_df.to_csv(ESTACOES_BASE_PATH, index=False, encoding="utf-8")
    return station_df


def load_accident_source() -> pd.DataFrame:
    """Carrega os sinistros usados para escolher as estações mais próximas."""
    existing_df = globals().get("df")
    if isinstance(existing_df, pd.DataFrame) and {"latitude", "longitude"}.issubset(existing_df.columns):
        return existing_df.copy()

    return pd.read_csv(SINISTROS_SOURCE_PATH, sep=";", encoding="latin1", dtype=str)


def to_numeric_coordinate(series: pd.Series) -> pd.Series:
    """Converte coordenadas com vírgula decimal para número."""
    return pd.to_numeric(
        series.astype(str).str.replace(",", ".", regex=False).str.strip(),
        errors="coerce",
    )


def haversine_distances_km(
    latitude: float,
    longitude: float,
    station_latitudes: np.ndarray,
    station_longitudes: np.ndarray,
) -> np.ndarray:
    """Calcula distâncias Haversine em quilômetros para várias estações."""
    earth_radius_km = 6371.0088
    lat1 = np.radians(latitude)
    lon1 = np.radians(longitude)
    lat2 = np.radians(station_latitudes.astype(float))
    lon2 = np.radians(station_longitudes.astype(float))
    delta_lat = lat2 - lat1
    delta_lon = lon2 - lon1
    a = np.sin(delta_lat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(delta_lon / 2.0) ** 2
    return 2.0 * earth_radius_km * np.arcsin(np.sqrt(np.clip(a, 0.0, 1.0)))


def build_accident_station_mapping(station_catalog: pd.DataFrame, accident_df: pd.DataFrame) -> pd.DataFrame:
    """Relaciona cada sinistro válido com a estação mais próxima."""
    stations_with_coordinates = station_catalog.copy()
    stations_with_coordinates["latitude_num"] = to_numeric_coordinate(stations_with_coordinates["latitude"])
    stations_with_coordinates["longitude_num"] = to_numeric_coordinate(stations_with_coordinates["longitude"])
    stations_with_coordinates = stations_with_coordinates.dropna(subset=["latitude_num", "longitude_num"]).copy()

    if stations_with_coordinates.empty:
        raise RuntimeError("Nao foi possivel encontrar estações com coordenadas validas.")

    accidents_with_coordinates = accident_df.copy()
    accidents_with_coordinates["latitude_num"] = to_numeric_coordinate(accidents_with_coordinates["latitude"])
    accidents_with_coordinates["longitude_num"] = to_numeric_coordinate(accidents_with_coordinates["longitude"])
    accidents_with_coordinates = accidents_with_coordinates.dropna(subset=["latitude_num", "longitude_num"]).copy()

    mapping_rows: list[dict[str, object]] = []
    station_latitudes = stations_with_coordinates["latitude_num"].to_numpy(dtype=float)
    station_longitudes = stations_with_coordinates["longitude_num"].to_numpy(dtype=float)

    # Mantém um relacionamento persistido para evitar recalcular as distâncias em execuções futuras.
    for accident in accidents_with_coordinates.itertuples(index=False):
        accident_id = normalize_station_value(getattr(accident, "id", None))
        if not accident_id:
            continue

        distances = haversine_distances_km(
            float(accident.latitude_num),
            float(accident.longitude_num),
            station_latitudes,
            station_longitudes,
        )
        nearest_index = int(np.argmin(distances))
        nearest_station = stations_with_coordinates.iloc[nearest_index]
        mapping_rows.append(
            {
                "id": accident_id,
                "wmo": str(nearest_station["wmo"]),
                "distancia_km": float(distances[nearest_index]),
            }
        )

    mapping_df = pd.DataFrame(mapping_rows, columns=EXPECTED_MAPPING_COLUMNS)
    if not mapping_df.empty:
        print(
            f"Sinistros válidos: {len(mapping_df)} | "
            f"estações distintas: {mapping_df['wmo'].nunique()} | "
            f"distancia media: {mapping_df['distancia_km'].mean():.2f} km"
        )
    else:
        print("Nenhum sinistro válido com coordenadas foi encontrado.")
    return mapping_df


def load_or_build_accident_station_mapping(station_catalog: pd.DataFrame, accident_df: pd.DataFrame) -> pd.DataFrame:
    """Carrega sinistro_estacao_mais_proxima.csv ou gera o relacionamento do zero."""
    if SINISTRO_ESTACAO_MAIS_PROXIMA_PATH.exists():
        print(f"Cache hit: {SINISTRO_ESTACAO_MAIS_PROXIMA_PATH}")
        mapping_df = pd.read_csv(SINISTRO_ESTACAO_MAIS_PROXIMA_PATH, dtype={"id": str, "wmo": str})
        mapping_df = mapping_df.rename(columns=slugify_text)
        mapping_df = mapping_df[[column for column in EXPECTED_MAPPING_COLUMNS if column in mapping_df.columns]]
        if "distancia_km" in mapping_df.columns:
            mapping_df["distancia_km"] = pd.to_numeric(mapping_df["distancia_km"], errors="coerce")
        return mapping_df

    mapping_df = build_accident_station_mapping(station_catalog, accident_df)
    INMET_AGGREGATED_ROOT.mkdir(parents=True, exist_ok=True)
    mapping_df.to_csv(SINISTRO_ESTACAO_MAIS_PROXIMA_PATH, index=False, encoding="utf-8")
    return mapping_df


def load_or_build_final_stations(station_catalog: pd.DataFrame, accident_station_mapping: pd.DataFrame) -> pd.DataFrame:
    """Carrega estacoes_finais.csv ou recalcula a seleção a partir do relacionamento persistido."""
    if ESTACOES_FINAIS_PATH.exists():
        print(f"Cache hit: {ESTACOES_FINAIS_PATH}")
        return clean_station_catalog(pd.read_csv(ESTACOES_FINAIS_PATH, dtype=str))[EXPECTED_STATION_COLUMNS]

    selected_wmos = sorted(
        accident_station_mapping["wmo"].dropna().astype(str).unique().tolist()
    )
    final_stations = station_catalog.loc[station_catalog["wmo"].isin(selected_wmos)].copy()
    final_stations = final_stations[EXPECTED_STATION_COLUMNS]
    final_stations.to_csv(ESTACOES_FINAIS_PATH, index=False, encoding="utf-8")
    return final_stations


def build_processed_file_index() -> dict[str, Path]:
    """Indexa os CSVs processados por nome do arquivo para leitura rápida."""
    file_index: dict[str, Path] = {}
    for data_path in INMET_PROCESSED_ROOT.rglob("*_dados.csv"):
        file_index[data_path.name] = data_path
    return file_index


def parse_and_convert_station_history(data_path: Path, wmo: str) -> pd.DataFrame:
    """Normaliza um CSV bruto de estação para o formato final do histórico."""
    raw_df = pd.read_csv(data_path, encoding="latin1")
    raw_df = normalize_dataframe_columns(raw_df)

    if not {"data", "hora_utc"}.issubset(raw_df.columns):
        raise ValueError(f"Colunas de data ausentes em {data_path.name}")

    timestamps_utc = pd.to_datetime(
        raw_df["data"].astype(str).str.strip() + " " + raw_df["hora_utc"].astype(str).str.strip(),
        format="%Y/%m/%d %H%M UTC",
        errors="coerce",
        utc=True,
    )

    converted_df = raw_df.drop(columns=["data", "hora_utc"], errors="ignore").copy()
    converted_df.insert(0, "wmo", wmo)
    converted_df.insert(
        1,
        "data_hora",
        timestamps_utc.dt.tz_convert(BRASILIA_TZ).dt.strftime("%Y-%m-%d %H:%M:%S"),
    )
    converted_df = converted_df.dropna(subset=["data_hora"]).copy()

    for column in converted_df.columns:
        if column in {"wmo", "data_hora"}:
            continue
        converted_df[column] = pd.to_numeric(
            converted_df[column].astype(str).str.replace(",", ".", regex=False).str.strip(),
            errors="coerce",
        )

    ordered_columns = ["wmo", "data_hora"] + [column for column in converted_df.columns if column not in {"wmo", "data_hora"}]
    return converted_df.loc[:, ordered_columns]


def build_historico_from_catalog(selected_catalog: pd.DataFrame) -> pd.DataFrame:
    """Consolida apenas os arquivos das estações selecionadas."""
    if selected_catalog.empty:
        return pd.DataFrame(columns=["wmo", "data_hora"])

    file_index = build_processed_file_index()
    history_chunks: list[pd.DataFrame] = []
    missing_files: list[str] = []

    for station in selected_catalog.itertuples(index=False):
        source_file = normalize_station_value(getattr(station, "source_file", None))
        if not source_file:
            continue

        data_path = file_index.get(source_file)
        if data_path is None:
            missing_files.append(source_file)
            continue

        station_history = parse_and_convert_station_history(data_path, str(station.wmo))
        if not station_history.empty:
            history_chunks.append(station_history)

    if missing_files:
        print("[AVISO] Arquivos de histórico nao encontrados:")
        for missing_file in sorted(set(missing_files)):
            print(f"  - {missing_file}")

    if not history_chunks:
        return pd.DataFrame(columns=["wmo", "data_hora"])

    historico_df = pd.concat(history_chunks, ignore_index=True)
    historico_df = historico_df.sort_values(["wmo", "data_hora"], kind="stable").reset_index(drop=True)
    ordered_columns = ["wmo", "data_hora"] + [column for column in historico_df.columns if column not in {"wmo", "data_hora"}]
    return historico_df.loc[:, ordered_columns]


def load_or_build_historico(selected_catalog: pd.DataFrame) -> pd.DataFrame:
    """Carrega inmet_historico.csv ou gera a consolidação final do zero."""
    if INMET_HISTORICO_PATH.exists():
        print(f"Cache hit: {INMET_HISTORICO_PATH}")
        return pd.read_csv(INMET_HISTORICO_PATH)

    historico_df = build_historico_from_catalog(selected_catalog)
    historico_df.to_csv(INMET_HISTORICO_PATH, index=False, encoding="utf-8")
    return historico_df


INMET_AGGREGATED_ROOT.mkdir(parents=True, exist_ok=True)
station_catalog = load_station_catalog()
accident_df = load_accident_source()
accident_station_mapping = load_or_build_accident_station_mapping(station_catalog, accident_df)
final_station_catalog = load_or_build_final_stations(station_catalog, accident_station_mapping)
historico_df = load_or_build_historico(final_station_catalog)


Cache hit: data/datasets/inmet_aggregated/estacoes.csv
Cache hit: data/datasets/inmet_aggregated/sinistro_estacao_mais_proxima.csv
Cache hit: data/datasets/inmet_aggregated/estacoes_finais.csv
Cache hit: data/datasets/inmet_aggregated/inmet_historico.csv
Estacoes base: 594
Relacionamento sinistro-estacao: 72529
Estacoes finais: 481
Historico INMET: 4072224
Arquivo de relacionamento: data/datasets/inmet_aggregated/sinistro_estacao_mais_proxima.csv
Arquivo de estacoes finais: data/datasets/inmet_aggregated/estacoes_finais.csv
Arquivo de historico: data/datasets/inmet_aggregated/inmet_historico.csv


,wmo,data_hora,precipitaaao_total_horario_mm,pressao_atmosferica_ao_nivel_da_estacao_horaria_mb,pressao_atmosferica_max_na_hora_ant_aut_mb,pressao_atmosferica_min_na_hora_ant_aut_mb,radiacao_global_kj_ma2,temperatura_do_ar_bulbo_seco_horaria_ac,temperatura_do_ponto_de_orvalho_ac,temperatura_maxima_na_hora_ant_aut_ac,temperatura_manima_na_hora_ant_aut_ac,temperatura_orvalho_max_na_hora_ant_aut_ac,temperatura_orvalho_min_na_hora_ant_aut_ac,umidade_rel_max_na_hora_ant_aut,umidade_rel_min_na_hora_ant_aut,umidade_relativa_do_ar_horaria,vento_direaao_horaria_gr_a_gr,vento_rajada_maxima_m_s,vento_velocidade_horaria_m_s
0,A001,2024-12-31 21:00:00,0.0,886.1,886.1,885.5,NaN,20.8,19.5,20.9,20.7,19.5,19.2,92.0,90.0,92.0,8.0,3.6,1.8
1,A001,2024-12-31 22:00:00,0.0,886.3,886.4,886.0,NaN,20.7,19.3,20.8,20.5,19.5,19.3,93.0,92.0,92.0,4.0,3.4,1.9
2,A001,2024-12-31 23:00:00,0.0,886.7,886.7,886.2,NaN,20.6,19.1,20.8,20.5,19.3,19.0,92.0,91.0,91.0,3.0,3.7,1.4
3,A001,2025-01-01 00:00:00,0.0,886.4,886.7,886.4,NaN,20.3,18.9,20.6,20.1,19.1,18.7,92.0,91.0,92.0,345.0,3.0,1.1
4,A001,2025-01-01 01:00:00,0.0,885.9,886.4,885.9,NaN,19.8,18.9,20.3,19.7,19.1,18.8,95.0,92.0,94.0,228.0,2.0,0.3


In [10]:
print(f"Arquivo de estacoes finais: {ESTACOES_FINAIS_PATH}")
print(f"Estacoes base: {len(station_catalog)}")
print(f"Estacoes finais: {len(final_station_catalog)}")
final_station_catalog.head()

Arquivo de estacoes finais: data/datasets/inmet_aggregated/estacoes_finais.csv
Estacoes base: 594
Estacoes finais: 481


,wmo,regiao,uf,estacao,latitude,longitude,altitude,data_de_fundacao,source_file
0,A001,CO,DF,BRASILIA,"-15,78944444","-47,92583332","1160,96",07/05/00,inmet_CO_DF_A001_BRASILIA_01-01-2025_A_31-12-2...
1,A042,CO,DF,BRAZLANDIA,"-15,59972221","-48,1311111",1143,19/07/17,inmet_CO_DF_A042_BRAZLANDIA_01-01-2025_A_31-12...
2,A045,CO,DF,AGUAS EMENDADAS,"-15,59638888","-47,62583332","1030,36",03/10/08,inmet_CO_DF_A045_AGUAS-EMENDADAS_01-01-2025_A_...
3,A046,CO,DF,GAMA (PONTE ALTA),"-15,93527777","-48,13749999",990,01/10/14,inmet_CO_DF_A046_GAMA-(PONTE-ALTA)_01-01-2025_...
4,A047,CO,DF,PARANOA (COOPA-DF),"-16,01222222","-47,5575",1043,07/02/17,inmet_CO_DF_A047_PARANOA-(COOPA-DF)_01-01-2025...


In [11]:
print(f"Arquivo de relacionamento: {SINISTRO_ESTACAO_MAIS_PROXIMA_PATH}")
print(f"Relacionamento sinistro-estacao: {len(accident_station_mapping)}")
accident_station_mapping.head()

Arquivo de relacionamento: data/datasets/inmet_aggregated/sinistro_estacao_mais_proxima.csv
Relacionamento sinistro-estacao: 72529


,id,wmo,distancia_km
0,652493,A701,8.170926
1,652519,A370,27.364351
2,652522,A842,26.579262
3,652544,B806,12.541341
4,652549,A506,50.238341


In [12]:
print(f"Arquivo de historico: {INMET_HISTORICO_PATH}")
print(f"Historico INMET: {len(historico_df)}")
historico_df.head()

Arquivo de historico: data/datasets/inmet_aggregated/inmet_historico.csv
Historico INMET: 4072224


,wmo,data_hora,precipitaaao_total_horario_mm,pressao_atmosferica_ao_nivel_da_estacao_horaria_mb,pressao_atmosferica_max_na_hora_ant_aut_mb,pressao_atmosferica_min_na_hora_ant_aut_mb,radiacao_global_kj_ma2,temperatura_do_ar_bulbo_seco_horaria_ac,temperatura_do_ponto_de_orvalho_ac,temperatura_maxima_na_hora_ant_aut_ac,temperatura_manima_na_hora_ant_aut_ac,temperatura_orvalho_max_na_hora_ant_aut_ac,temperatura_orvalho_min_na_hora_ant_aut_ac,umidade_rel_max_na_hora_ant_aut,umidade_rel_min_na_hora_ant_aut,umidade_relativa_do_ar_horaria,vento_direaao_horaria_gr_a_gr,vento_rajada_maxima_m_s,vento_velocidade_horaria_m_s
0,A001,2024-12-31 21:00:00,0.0,886.1,886.1,885.5,NaN,20.8,19.5,20.9,20.7,19.5,19.2,92.0,90.0,92.0,8.0,3.6,1.8
1,A001,2024-12-31 22:00:00,0.0,886.3,886.4,886.0,NaN,20.7,19.3,20.8,20.5,19.5,19.3,93.0,92.0,92.0,4.0,3.4,1.9
2,A001,2024-12-31 23:00:00,0.0,886.7,886.7,886.2,NaN,20.6,19.1,20.8,20.5,19.3,19.0,92.0,91.0,91.0,3.0,3.7,1.4
3,A001,2025-01-01 00:00:00,0.0,886.4,886.7,886.4,NaN,20.3,18.9,20.6,20.1,19.1,18.7,92.0,91.0,92.0,345.0,3.0,1.1
4,A001,2025-01-01 01:00:00,0.0,885.9,886.4,885.9,NaN,19.8,18.9,20.3,19.7,19.1,18.8,95.0,92.0,94.0,228.0,2.0,0.3
